# 03 — Statistical Analysis and Logistic Regression

Synthetic healthcare portfolio dataset only.

In [ ]:
import pandas as pd
from pathlib import Path
df=pd.read_csv(Path('../data/healthcare_python_cleaned.csv'))

In [ ]:
from scipy.stats import chi2_contingency,ttest_ind,pearsonr
print(chi2_contingency(pd.crosstab(df.Gender,df.Readmission30d)))
print(ttest_ind(df.loc[df.Readmission30d==1,'BMI'],df.loc[df.Readmission30d==0,'BMI'],equal_var=False))
print(pearsonr(df.Age,df.SystolicBP))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
num=['Age','BMI','SystolicBP','DiastolicBP','LengthOfStayDays']; cat=['Gender','Diagnosis']
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
pipe=Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))])
X=df[num+cat]; y=df.Readmission30d
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); prob=pipe.predict_proba(Xte)[:,1]
print(confusion_matrix(yte,pred)); print(classification_report(yte,pred,zero_division=0)); print('ROC-AUC',roc_auc_score(yte,prob))

In [ ]:
print('Warning: outcome and predictors are synthetic; this model is not clinically valid.')